# Workflow 07_A: Basic RAG Baseline (1 LLM call, no agent/tools)

**Architecture**
1. Receive user query.
2. Retrieve top-k product documents from `ProductRetriever` (ChromaDB + rerank).
3. Build a single prompt: system + retrieved context + question.
4. Call `LLMService.call_gemini` once with `stream=False`.
5. Return the LLM answer.

**Limitation vs Agentic RAG**
- No real-time SQL filters (price, stock, exact models).
- No tool calls (orders, policies, compare, multi-turn context).
- Pure vector context may miss exact specs/prices.


In [ ]:
# Setup path and imports
import sys, os, json, time
from pathlib import Path

notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

from configs.setting import settings
from configs.GetConfig import config
from src.LLMService import LLMService
from src.h_evaluation.benchmark_evaluator import BenchmarkEvaluator
from src.c_retrieval.product_retriever import ProductRetriever

llm_service = LLMService(settings, config)
product_retriever = ProductRetriever(config, settings)
model = config.llm.google.available[0]
print("Model:", model)


In [ ]:
def extract_text(response):
    """Extract text from LLMService response (list of chunks or single response)."""
    if not response:
        return ""
    if isinstance(response, list):
        parts = []
        for chunk in response:
            if getattr(chunk, "text", None):
                parts.append(chunk.text)
            elif getattr(chunk, "candidates", None):
                for cand in chunk.candidates:
                    content = getattr(cand, "content", None)
                    if content:
                        for part in getattr(content, "parts", []):
                            parts.append(getattr(part, "text", ""))
        return "".join(parts)
    return getattr(response, "text", "")

def basic_rag_answer(query, k=3, system_prompt=None):
    """Basic RAG: retrieve -> single LLM call."""
    t0 = time.time()
    results = product_retriever.retrieve(query_text=query, limit=k)
    contexts = []
    for r in results:
        doc = r.get("document", "")
        meta = r.get("metadata", {})
        contexts.append(f"Product: {meta.get('name','')}\nBrand: {meta.get('brand','')}\n{doc}")
    ctx_text = "\n\n---\n\n".join(contexts) if contexts else "Không tìm thấy thông tin liên quan."

    if system_prompt is None:
        system_prompt = (
            "Bạn là trợ lý bán hàng điện thoại/laptop. Dùng thông tin trong phần CONTEXT để trả lời câu hỏi. "
            "Nếu thông tin không đủ, hãy nói rõ. Không bịa đặt. Trả lời ngắn gọn, đúng trọng tâm."
        )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"CONTEXT:\n{ctx_text}\n\nCÂU HỎI: {query}\n\nTrả lời:"}
    ]
    response = llm_service.call_gemini(model=model, messages=messages, tools=None, stream=False)
    answer = extract_text(response)
    return {
        "final_answer": answer,
        "tool_outputs": contexts,
        "actual_tool_calls": [{"tool": "retriever", "args": {"query": query, "limit": k}}],
        "latency": time.time() - t0,
        "total_tokens": 0,
    }


In [ ]:
# Test on 3 sample queries from the benchmark
bench_path = Path("/home/ubuntu/benchmark_results/ecommerce_benchmark_20each_1785642048.jsonl")
samples = [json.loads(l) for l in bench_path.read_text(encoding='utf-8').splitlines()[:3]]

for rec in samples:
    q = rec.get("question", "") or rec["turns"][0]["question"]
    print(f"\nQ: {q}")
    res = basic_rag_answer(q, k=3)
    print(f"A: {res['final_answer'][:300]}...")


In [ ]:
def flatten_benchmark(path, limit=None):
    rows = []
    with open(path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            rec = json.loads(line)
            if rec.get("turns"):
                for j, turn in enumerate(rec["turns"]):
                    rows.append({
                        "id": f"{rec['id']}_turn{j+1}",
                        "category": rec.get("category", ""),
                        "question": turn["question"],
                        "expected_tool_calls": turn.get("expected_tool_calls", []),
                        "ground_truth": turn.get("ground_truth", {}),
                    })
            else:
                rows.append({
                    "id": rec["id"],
                    "category": rec.get("category", ""),
                    "question": rec["question"],
                    "expected_tool_calls": rec.get("expected_tool_calls", []),
                    "ground_truth": rec.get("ground_truth", {}),
                })
    return rows

# Run on the first N records and save raw results
BENCH_LIMIT = 10

raw_results = []
for rec in flatten_benchmark("/home/ubuntu/benchmark_results/ecommerce_benchmark_20each_1785642048.jsonl", limit=BENCH_LIMIT):
    try:
        res = basic_rag_answer(rec["question"], k=3)
        raw_results.append({
            "id": rec["id"],
            "category": rec["category"],
            "question": rec["question"],
            "workflow": "07_A_basic_rag",
            "final_answer": res["final_answer"],
            "tool_outputs": res["tool_outputs"],
            "actual_tool_calls": res["actual_tool_calls"],
            "ground_truth": rec["ground_truth"],
            "expected_tool_calls": rec["expected_tool_calls"],
            "latency": res["latency"],
            "total_tokens": res["total_tokens"],
        })
    except Exception as e:
        print(f"Error on {rec['id']}: {e}")

out_path = Path("benchmark_results/raw_07_A_basic_rag.jsonl")
out_path.parent.mkdir(exist_ok=True, parents=True)
with open(out_path, "w", encoding="utf-8") as f:
    for r in raw_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(raw_results)} rows to {out_path}")


In [ ]:
# Evaluate the raw results (custom judge only; set use_ragas=True if API quota allows)
evaluator = BenchmarkEvaluator(judge_provider='groq', use_ragas=False)
report = evaluator.evaluate(raw_results, output_dir='benchmark_results')
evaluator.print_table(report)
